# Part 5, notebook 6 of 6: what it costs, the live run, and what is not covered

**The analogy.** A security guard has a price: do visitors wait longer at the door? Then one full day of real visitors with the guard in place. Then, honestly, a list of the burglars the guard is not trained for.

## 1. What the container costs

10 sessions per set-up, 30 trivial cells per session. Start-up is the time to open a session; per-cell is the time for `x = np.arange(1000).sum()`. This runs live and takes about 30 s.

In [1]:
import sys, time, json
from pathlib import Path
sys.path.insert(0, "../minimal_loop")
import numpy as np
from docker_session import DockerSession
from session4 import Session4

IMG = np.zeros((224, 224), np.uint8)
for name, cls in (("unsandboxed Jupyter kernel", Session4), ("container", DockerSession)):
    starts, cells = [], []
    for _ in range(10):
        t = time.monotonic(); s = cls(IMG, timeout_sec=10, with_show=True, with_tools=True); starts.append(time.monotonic() - t)
        for _ in range(30):
            t = time.monotonic(); s.run("x = np.arange(1000).sum()"); cells.append(time.monotonic() - t)
        s.close()
    print(f"{name:28s} start: median {np.median(starts)*1000:6.0f} ms | per trivial cell: median {np.median(cells)*1000:5.1f} ms, p95 {np.percentile(cells, 95)*1000:5.1f} ms")

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


...

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


unsandboxed Jupyter kernel   start: median    517 ms | per trivial cell: median   7.3 ms, p95   9.5 ms


container                    start: median    558 ms | per trivial cell: median   0.3 ms, p95   0.9 ms


The container answers a cell faster (it talks over a pipe to a plain process; the Jupyter kernel goes through ZeroMQ and a message protocol) and starts about as fast. In a loop where the model takes seconds per step, neither matters.

## 2. The live run: 12 DEV images, model code inside the container

**This notebook does not re-run the model.** The live run needs the local model server and about two minutes of model time; its result is read from the saved file `out/run_dev4_Csbx.json`, next to the two unsandboxed condition-C runs of Part 4. It is **one run**: it shows the sandbox works end to end, and it says nothing reliable about accuracy, because Part 4 showed one run is noise (mean errors of 2.63 and 0.26 pp for the same set-up).

In [2]:
ROOT = Path("..")
runs = {"C (Part 4, run 1, unsandboxed)": "C", "C (Part 4, run 2, unsandboxed)": "Cr2", "C (Part 5, container)": "Csbx"}
print(f"{'set-up':34s} {'n':>2s} {'REVIEW':>6s} {'MAE pp':>7s} {'median':>7s} {'verdicts ok':>11s} {'false PASS':>10s} {'false FAIL':>10s} {'steps':>5s} {'tokens':>7s} {'sec':>5s}")
for name, tag in runs.items():
    rows = json.load(open(ROOT / f"out/run_dev4_{tag}.json"))["rows"]
    e = np.array([r["abs_err"] for r in rows if r["abs_err"] is not None])
    print(f"{name:34s} {len(rows):2d} {sum(r['agent_status'] == 'REVIEW' for r in rows):6d} {e.mean():7.2f} {np.median(e):7.2f} "
          f"{sum(r['agent_status'] == r['gt_status'] for r in rows):>8d}/{len(rows)} {sum(r['gt_status'] == 'FAIL' and r['agent_status'] == 'PASS' for r in rows):10d} "
          f"{sum(r['gt_status'] == 'PASS' and r['agent_status'] == 'FAIL' for r in rows):10d} {np.mean([r['steps'] for r in rows]):5.1f} {np.mean([r['tokens'] for r in rows]):7.0f} {np.mean([r['seconds'] for r in rows]):5.0f}")

set-up                              n REVIEW  MAE pp  median verdicts ok false PASS false FAIL steps  tokens   sec
C (Part 4, run 1, unsandboxed)     12      0    2.63    0.18        8/12          1          3   7.8   27293   110
C (Part 4, run 2, unsandboxed)     12      0    0.26    0.13       11/12          1          0   6.6   19595    91
C (Part 5, container)              12      0    0.19    0.17       11/12          0          1   7.1   25110   127


In [3]:
rows = json.load(open(ROOT / "out/run_dev4_Csbx.json"))["rows"]
print("sandboxed run, per image (gt = ground truth void ratio in percent of the part):")
for r in rows:
    print(f"  {r['image']}  gt {r['gt_pct']:5.2f}  agent {('-' if r['agent_pct'] is None else format(r['agent_pct'], '.2f')):>6s}  {r['agent_status']:6s}  steps {r['steps']:2d}  {r['reason']}")

sandboxed run, per image (gt = ground truth void ratio in percent of the part):
  val_00000  gt  1.62  agent   1.72  FAIL    steps 10  rule: 1.72 > 1.50
  val_00001  gt  0.61  agent   0.31  PASS    steps  5  rule: 0.31 <= 1.50
  val_00002  gt  0.96  agent   1.04  PASS    steps  5  rule: 1.04 <= 1.50
  val_00003  gt  0.65  agent   0.48  PASS    steps  5  rule: 0.48 <= 1.50
  val_00004  gt  0.46  agent   0.42  PASS    steps  4  rule: 0.42 <= 1.50
  val_00005  gt  0.17  agent   0.00  PASS    steps  8  rule: 0.00 <= 1.50
  val_00006  gt  0.49  agent   0.28  PASS    steps  7  rule: 0.28 <= 1.50
  val_00007  gt  1.15  agent   0.99  PASS    steps  4  rule: 0.99 <= 1.50
  val_00008  gt  5.02  agent   5.47  FAIL    steps  8  rule: 5.47 > 1.50
  val_00009  gt  1.38  agent   1.79  FAIL    steps 13  rule: 1.79 > 1.50
  val_00010  gt  1.81  agent   1.63  FAIL    steps 10  rule: 1.63 > 1.50
  val_00011  gt  0.00  agent   0.00  PASS    steps  6  rule: 0.00 <= 1.50


**How to read it.** 12 of 12 images produced a measurement, 0 ended in REVIEW, 0 sandbox errors. The accuracy looks better than Part 4's unsandboxed runs, but **do not read that as "the sandbox helps"**: it is one run of a noisy model, and the sandbox cannot change what the model decides. The illustrative 1.5 % limit is a made-up rule for the series, not a real inspection standard.

## 3. What the sandbox does not protect against

| Not covered | Why it matters | What would address it |
|---|---|---|
| **Container escapes** | A container shares the host kernel; a kernel bug can break out | gVisor or a micro-VM for code from the internet |
| **Whoever can talk to Docker** | My user is in the `docker` group, which is root-equivalent | The sandbox protects the host from the model, not from me |
| **Wasting resources inside the limits** | One CPU for the timeout, 64 MB of disk, a 14 s window | A stricter budget is a policy choice |
| **Wrong numbers** | A sandbox cannot tell whether `28.99` is right | A plausibility check in the rule layer (planned for Part 7) |
| **The image supply chain** | Base image and 3 pip packages are trusted as downloaded; no pinned digests | Pin digests, scan the image |
| **Untested** | seccomp beyond Docker's default, timing side channels, the GPU (none in the container), any attack not in the list | An audit, not a smoke test |

**Where this differs from the paper.** The paper's safety layer is an AST filter. This series uses none: the boundary is the operating system. Whether an AST filter adds anything *on top* of the container is not measured.

**What I did not do.** One sandboxed live run, no accuracy claim. No held-out data (Part 6 uses it, once). No hardened-variant comparison (gVisor, rootless Docker).